<a href="https://colab.research.google.com/github/Mychal003/Fake-News-Detector/blob/Maks/Preprocessing_Maks_poprawka.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install -U numpy pandas
!pip -q install -U spacy thinc \
  cymem preshed blis srsly \
  catalogue murmurhash unidecode nltk
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 75.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import os
import spacy
import re
import html
import string
import pandas as pd
import matplotlib.pyplot as plt
from unidecode import unidecode

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
df = pd.read_csv('/content/drive/MyDrive/WELFake_Dataset.csv')

In [ ]:
print(df.columns.tolist())

['Unnamed: 0', 'title', 'text', 'label']


In [ ]:
title_col = next((c for c in df.columns if c.lower() == "title"), None)
text_col  = next((c for c in df.columns if c.lower() == "text"),  None)
label_col = next((c for c in df.columns if "label" in c.lower()), None)
df[title_col] = df[title_col].fillna("") if title_col else ""
df[text_col]  = df[text_col].fillna("")  if text_col  else ""
df["title_text"] = (
    (df[title_col] if title_col else "") + " " + (df[text_col] if text_col else "")
).str.strip()

Czyszczenie

In [ ]:
URL_RE  = re.compile(r"(https?://\S+|www\.\S+)")
HTML_RE = re.compile(r"<.*?>")
WS_RE   = re.compile(r"\s+")

In [ ]:
def clean_basic(x: str) -> str:
    if x is None: return ""
    x = html.unescape(str(x))
    x = URL_RE.sub(" ", x)
    x = HTML_RE.sub(" ", x)
    x = x.lower()
    x = unidecode(x)
    x = x.translate(str.maketrans("", "", string.punctuation))
    x = WS_RE.sub(" ", x).strip()
    return x

In [ ]:
df["title_text_clean"] = df["title_text"].map(clean_basic)

lematyzacja + stopwords (spaCy)

In [ ]:
nlp = spacy.load("en_core_web_sm", disable=["ner","parser","textcat"])

In [ ]:
def spacy_lemmatize_stop(texts):
    out = []
    for doc in nlp.pipe(texts, batch_size=512):
        toks = [t.lemma_ for t in doc if not t.is_stop and not t.is_punct and not t.is_space]
        out.append(" ".join(toks))
    return out

In [ ]:
df["clean_lemma"] = spacy_lemmatize_stop(df["title_text_clean"])
df[["title_text", "clean_lemma"]].head()

,title_text,clean_lemma
0,LAW ENFORCEMENT ON HIGH ALERT Following Threat...,law enforcement high alert follow threat cop w...
1,Did they post their votes for Hillary already?,post vote hillary
2,UNBELIEVABLE! OBAMA’S ATTORNEY GENERAL SAYS MO...,unbelievable obamas attorney general say charl...
3,"Bobby Jindal, raised Hindu, uses story of Chri...",bobby jindal raise hindu use story christian c...
4,SATAN 2: Russia unvelis an image of its terrif...,satan 2 russia unveli image terrifying new sup...


In [ ]:
from transformers import AutoTokenizer
from tqdm.auto import tqdm

In [ ]:
tok = AutoTokenizer.from_pretrained("distilbert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
texts = df["title_text"].astype(str).values

In [ ]:
MAX_LEN = 128
BATCH = 512
OUT_DIR = "enc_out"
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
import numpy as np
import gc
part = 0
for i in tqdm(range(0, len(texts), BATCH)):
    batch = texts[i:i+BATCH]

    enc = tok(
        list(batch),
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="np"
    )
    np.savez_compressed(
        f"{OUT_DIR}/enc_part_{part:04d}.npz",
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"]
    )
    part += 1
    del enc, batch
    gc.collect()

print(f"Zapisano {part} plików w {OUT_DIR}/")

  0%|          | 0/141 [00:00<?, ?it/s]

Zapisano 141 plików w enc_out/


In [ ]:
sample = df["title_text"].astype(str).head(1000).tolist()
enc_small = tok(sample, padding="max_length", truncation=True, max_length=MAX_LEN, return_tensors="np")
enc_small["input_ids"].shape, enc_small["attention_mask"].shape

((1000, 128), (1000, 128))

In [ ]:
save_cols = ["clean_lemma"]
if "label" in df.columns:
    save_cols.append("label")
df[save_cols].to_csv("train_clean.csv", index=False)

In [ ]:
def tok_count(s): return len(str(s).split())
before = df["title_text"].astype(str)
after  = df["clean_lemma"].astype(str)

print("Średnia długość (przed) :", before.map(tok_count).mean())
print("Średnia długość (po)    :", after.map(tok_count).mean())
print("Min/Max (po):", after.map(tok_count).min(), "/", after.map(tok_count).max())
print("Przykład przed -> po:\n", before.iloc[0], "\n→", after.iloc[0])

Średnia długość (przed) : 552.7242493137771
Średnia długość (po)    : 294.83587489949264
Min/Max (po): 0 / 19205
Przykład przed -> po:
 LAW ENFORCEMENT ON HIGH ALERT Following Threats Against Cops And Whites On 9-11By #BlackLivesMatter And #FYF911 Terrorists [VIDEO] No comment is expected from Barack Obama Members of the #FYF911 or #FukYoFlag and #BlackLivesMatter movements called for the lynching and hanging of white people and cops. They encouraged others on a radio show Tuesday night to  turn the tide  and kill white people and cops to send a message about the killing of black people in America.One of the F***YoFlag organizers is called  Sunshine.  She has a radio blog show hosted from Texas called,  Sunshine s F***ing Opinion Radio Show. A snapshot of her #FYF911 @LOLatWhiteFear Twitter page at 9:53 p.m. shows that she was urging supporters to  Call now!! #fyf911 tonight we continue to dismantle the illusion of white Below is a SNAPSHOT Twitter Radio Call Invite   #FYF911The radio 

**Co usunięto**  
   - URL i tagi HTML   
   - Interpunkcja, wielkie litery, diakrytyki
   - Stopwords – słowa częste, ale mało informatywne.  

**Lematyzacja/Stemming**  
   - Lematyzacja sprowadza słowa do form podstawowych  
   - Dzięki temu zmniejsza się słownik, zwiększa gęstość cech i poprawia generalizacja modelu.